# 04B — Études Optuna par `evaluation_track`

Ce notebook réalise les tâches 29 et 30. Il crée un registre de huit études reproductibles, y compris une étude vide explicite lorsqu'un track ne possède aucun domaine calibré. Optuna ne suggère que `domain_config_id` et réutilise exclusivement les métriques exhaustives de 04A : aucun spectre n'est chargé, aucun modèle ou seuil n'est réajusté et les batches 3-4 sont inaccessibles à l'objectif.

Les tracks soutenus sont comparés au front protocolaire 04A ; les tracks non soutenus mais calculables restent des études diagnostiques. Les configurations calculables hors garde-fous conservent leurs objectifs. Seules les erreurs techniques, métriques non finies et violations structurellement impossibles sont prunées. Aucun score composite, quota ou filtre de diversité n'est utilisé.

La dernière section gèle aussi le plan d'ablation apparié avant la prochaine exécution 8-tracks sur le batch 3. Les traitements exploratoires legacy du batch 3 sont déclarés dans la provenance.

## A — Initialisation et chemins centralisés

In [1]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import optuna
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Launch the notebook from the project root or notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 24)
optuna.logging.set_verbosity(optuna.logging.WARNING)

from src import experiment_config as expcfg
from src.protocol_governance import sha256_file
from src.utils import load_parquet, save_parquet
from src.workflows.protocol_audit import assert_no_forbidden_score_columns
from src.workflows.simca_optuna import (
    build_optuna_pareto_candidates,
    build_optuna_search_efficiency_audit,
    build_optuna_search_plan_hash,
    build_optuna_study_registry,
    build_preregistered_ablation_plan,
    close_optuna_study,
    make_optuna_binary_pareto_objective,
    optuna_trials_dataframe,
)

%load_ext autoreload
%autoreload 2

print("Python:", platform.python_version())
print("Optuna:", optuna.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)

Python: 3.14.6
Optuna: 4.9.0
PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
USE_WAVELENGTH_WINDOW = expcfg.USE_WAVELENGTH_WINDOW
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
GRID_RESULTS_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
DOMAIN_RESULTS_DIR = PROJECT_ROOT / "results" / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_OPTUNA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_PATHS = {
    name: GRID_RESULTS_DIR / filename
    for name, filename in expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES.items()
}
PROJECTION_ELIGIBILITY_PATH = DOMAIN_RESULTS_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["projection_eligibility"]
SPATIAL_LOCK_PATH = DOMAIN_RESULTS_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["spatial_postprocessing_lock"]
OUTPUT_PATHS = {
    name: OUTPUT_DIR / filename
    for name, filename in expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES.items()
}
STORAGE_PATH = OUTPUT_DIR / expcfg.SIMCA_OPTUNA_STORAGE_FILENAME

print("04A source:", GRID_RESULTS_DIR)
print("04B output:", OUTPUT_DIR)
print("Lookup-only 04A metrics:", expcfg.SIMCA_OPTUNA_REUSE_GRID_METRICS)

04A source: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v3_non_noisy_all
04B output: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all
Lookup-only 04A metrics: True


## B — Domaine 04A, fronts exhaustifs et provenance

In [3]:
required_inputs = [
    GRID_PATHS["configurations"],
    GRID_PATHS["threshold_metrics"],
    GRID_PATHS["pareto_reference"],
    GRID_PATHS["technical_audit"],
    GRID_PATHS["protocol"],
    PROJECTION_ELIGIBILITY_PATH,
    SPATIAL_LOCK_PATH,
]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing upstream inputs: {missing_inputs}")
if not expcfg.SIMCA_OPTUNA_REUSE_GRID_METRICS:
    raise RuntimeError("04B is lookup-only: SIMCA_OPTUNA_REUSE_GRID_METRICS must remain True.")

grid_protocol = json.loads(GRID_PATHS["protocol"].read_text(encoding="utf-8"))
grid_output_keys = ("configurations", "threshold_metrics", "pareto_reference", "technical_audit")
for key in grid_output_keys:
    expected_hash = str(grid_protocol["output_sha256"][key])
    actual_hash = sha256_file(GRID_PATHS[key])
    if actual_hash != expected_hash:
        raise RuntimeError(f"04A {key} hash mismatch.")

grid_configurations_df = load_parquet(GRID_PATHS["configurations"])
grid_threshold_metrics_df = load_parquet(GRID_PATHS["threshold_metrics"])
grid_pareto_reference_df = load_parquet(GRID_PATHS["pareto_reference"])
grid_technical_audit_df = load_parquet(GRID_PATHS["technical_audit"])
projection_eligibility_df = load_parquet(PROJECTION_ELIGIBILITY_PATH)
spatial_lock = json.loads(SPATIAL_LOCK_PATH.read_text(encoding="utf-8"))

upstream_protocol_hash = str(grid_protocol["protocol_hash"])
if set(grid_configurations_df["protocol_hash"].astype(str)) != {upstream_protocol_hash}:
    raise RuntimeError("04A configurations do not match grid_protocol.json.")
if set(projection_eligibility_df["protocol_hash"].astype(str)) != {upstream_protocol_hash}:
    raise RuntimeError("03C eligibility does not match the 04A parent protocol.")
if str(spatial_lock["protocol_hash"]) != upstream_protocol_hash:
    raise RuntimeError("03C spatial lock does not match the 04A parent protocol.")
if projection_eligibility_df["evaluation_track"].nunique() != 8:
    raise RuntimeError("03C must report exactly the eight evaluation tracks.")
if set(grid_protocol["calibration_batches"]) != set(expcfg.INTERNAL_CALIBRATION_BATCHES):
    raise RuntimeError("04A calibration batches differ from the central contract.")
if not set(expcfg.INTERNAL_CALIBRATION_FORBIDDEN_BATCHES).issubset(set(grid_protocol["forbidden_batches"])):
    raise RuntimeError("04A does not explicitly forbid every forbidden batch.")

grid_configurations_df = grid_configurations_df.merge(
    projection_eligibility_df[["evaluation_track", "eligibility_status", "eligibility_reason"]],
    on="evaluation_track",
    how="left",
    validate="many_to_one",
)
if grid_configurations_df["eligibility_status"].isna().any():
    raise RuntimeError("A 04A configuration has no explicit 03C eligibility status.")
if grid_configurations_df["calibration_id"].duplicated().any():
    raise RuntimeError("04A main domain must contain one row per calibration_id.")
if grid_configurations_df["domain_config_id"].duplicated().any():
    raise RuntimeError("Every representative domain_config_id must be unique.")

source_hashes = {
    "04A_configurations": sha256_file(GRID_PATHS["configurations"]),
    "04A_threshold_metrics": sha256_file(GRID_PATHS["threshold_metrics"]),
    "04A_pareto_reference": sha256_file(GRID_PATHS["pareto_reference"]),
    "04A_technical_audit": sha256_file(GRID_PATHS["technical_audit"]),
    "03C_projection_eligibility": sha256_file(PROJECTION_ELIGIBILITY_PATH),
    "03C_spatial_lock": sha256_file(SPATIAL_LOCK_PATH),
}
search_plan_hash = build_optuna_search_plan_hash(grid_configurations_df, source_hashes)
study_registry_df = build_optuna_study_registry(
    grid_configurations_df,
    projection_eligibility_df,
    results_tag=RESULTS_TAG,
    search_plan_hash=search_plan_hash,
)
display(study_registry_df[["track_id", "evaluation_track", "n_domain_configurations", "study_scope", "study_status", "study_seed"]])
print("Search plan hash:", search_plan_hash)

,track_id,evaluation_track,n_domain_configurations,study_scope,study_status,study_seed
0,E1,object_train__object_projection__2way,10,protocol,runnable,42
1,E2,object_train__object_projection__3way,402,protocol,runnable,43
2,E3,object_train__pixel_projection__2way,0,unsupported_empty,not_runnable_no_domain,44
3,E4,object_train__pixel_projection__3way,220,diagnostic_only,runnable_diagnostic_only,45
4,E5,pixel_train__object_projection__2way,1,protocol,runnable,46
5,E6,pixel_train__object_projection__3way,10,protocol,runnable,47
6,E7,pixel_train__pixel_projection__2way,7,protocol,runnable,48
7,E8,pixel_train__pixel_projection__3way,397,diagnostic_only,runnable_diagnostic_only,49


Search plan hash: d55c183926ce6a33b6b7f057ac9d073d08af08d49ece11fe5d47985fbb58beca


## C — Huit études multiobjectifs reproductibles

La seed de chaque étude dépend de sa position stable dans le registre E1–E8, jamais de l'ordre des lignes disponibles. Une base existante n'est reprise que si son `search_plan_hash` est identique. Les seules suggestions autorisées sont les valeurs catégorielles de `domain_config_id`.

In [4]:
trial_parts = []
for study_row in study_registry_df.itertuples(index=False):
    track = str(study_row.evaluation_track)
    track_domain_df = grid_configurations_df.loc[
        grid_configurations_df["evaluation_track"].astype(str).eq(track)
    ].copy()
    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{STORAGE_PATH.as_posix()}",
        engine_kwargs={"connect_args": {"timeout": expcfg.SIMCA_OPTUNA_STORAGE_TIMEOUT_SECONDS}},
    )
    sampler = optuna.samplers.TPESampler(
        seed=int(study_row.study_seed),
        n_startup_trials=int(expcfg.SIMCA_OPTUNA_N_STARTUP_TRIALS),
        multivariate=bool(expcfg.SIMCA_OPTUNA_SAMPLER_MULTIVARIATE),
    )
    study = optuna.create_study(
        study_name=str(study_row.study_name),
        storage=storage,
        sampler=sampler,
        directions=expcfg.SIMCA_OPTUNA_OBJECTIVE_SPECS[track]["directions"],
        load_if_exists=expcfg.SIMCA_OPTUNA_LOAD_EXISTING_STUDY,
    )
    existing_hash = str(study.user_attrs.get("search_plan_hash", ""))
    if study.trials and existing_hash != search_plan_hash:
        close_optuna_study(study)
        raise RuntimeError(f"Study {study.study_name} belongs to another search plan.")
    study.set_user_attr("search_plan_hash", search_plan_hash)
    study.set_user_attr("evaluation_track", track)
    study.set_user_attr("study_scope", str(study_row.study_scope))
    study.set_user_attr("study_seed", int(study_row.study_seed))

    if str(study_row.study_status) != "not_runnable_no_domain":
        objective = make_optuna_binary_pareto_objective(
            object_db=None,
            image_db=None,
            allowed_domain=track_domain_df,
            calibration_folds=None,
            decision_mode=str(study_row.decision_mode),
            precomputed_metrics=grid_threshold_metrics_df,
            evaluation_track=track,
            study_scope=str(study_row.study_scope),
            study_seed=int(study_row.study_seed),
            search_plan_hash=search_plan_hash,
        )
        remaining_trials = max(0, int(study_row.trial_budget) - len(study.trials))
        if expcfg.SIMCA_OPTUNA_RUN and remaining_trials:
            study.optimize(
                objective,
                n_trials=remaining_trials,
                n_jobs=expcfg.SIMCA_OPTUNA_N_JOBS,
                show_progress_bar=expcfg.SIMCA_OPTUNA_SHOW_PROGRESS_BAR,
            )
    if any(set(trial.params) - {"domain_config_id"} for trial in study.trials):
        close_optuna_study(study)
        raise RuntimeError(f"Study {study.study_name} suggested a forbidden parameter.")
    current_trials_df = optuna_trials_dataframe(study)
    if not current_trials_df.empty:
        trial_parts.append(current_trials_df)
    print(
        f"{study_row.track_id}: {len(current_trials_df)} trials | "
        f"scope={study_row.study_scope} | status={study_row.study_status}"
    )
    close_optuna_study(study)

all_trials_raw_df = (
    pd.concat(trial_parts, ignore_index=True, sort=False)
    if trial_parts
    else pd.DataFrame()
)
if not all_trials_raw_df.empty and all_trials_raw_df.groupby("study_name")["evaluation_track"].nunique().gt(1).any():
    raise RuntimeError("An Optuna study mixes multiple evaluation tracks.")

c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


E1: 100 trials | scope=protocol | status=runnable
E2: 100 trials | scope=protocol | status=runnable
E3: 0 trials | scope=unsupported_empty | status=not_runnable_no_domain


c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


E4: 100 trials | scope=diagnostic_only | status=runnable_diagnostic_only
E5: 100 trials | scope=protocol | status=runnable


c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


E6: 100 trials | scope=protocol | status=runnable
E7: 100 trials | scope=protocol | status=runnable


c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


E8: 100 trials | scope=diagnostic_only | status=runnable_diagnostic_only


## D — Trials, fronts Optuna et efficacité face à l'exhaustif

In [5]:
optuna_pareto_candidates_df = build_optuna_pareto_candidates(all_trials_raw_df)
optuna_trials_df = all_trials_raw_df.reindex(columns=expcfg.SIMCA_OPTUNA_TRIAL_COLUMNS)
technical_statuses = set(expcfg.SIMCA_OPTUNA_TECHNICAL_PRUNE_STATUSES)
if optuna_trials_df.empty:
    optuna_errors_df = pd.DataFrame(columns=expcfg.SIMCA_OPTUNA_ERROR_COLUMNS)
else:
    error_mask = (
        optuna_trials_df["status"].astype(str).isin(technical_statuses)
        | optuna_trials_df["error_type"].fillna("").astype(str).ne("")
    )
    optuna_errors_df = optuna_trials_df.loc[
        error_mask, list(expcfg.SIMCA_OPTUNA_ERROR_COLUMNS)
    ].copy()

optuna_search_efficiency_df = build_optuna_search_efficiency_audit(
    grid_configurations_df,
    optuna_trials_df,
    pareto_reference=grid_pareto_reference_df,
    study_registry=study_registry_df,
)
if len(optuna_search_efficiency_df) != 8:
    raise RuntimeError("Search efficiency must contain exactly the eight tracks.")
if not optuna_search_efficiency_df["exhaustive_reference_retained"].all():
    raise RuntimeError("An exhaustive 04A reference was marked for removal.")

display(
    optuna_trials_df.groupby(
        ["track_id", "state", "status"], dropna=False, as_index=False
    ).agg(n_trials=("trial_number", "count"))
)
display(
    optuna_pareto_candidates_df.groupby(
        ["track_id", "study_scope"], as_index=False
    ).agg(
        n_diagnostic=("diagnostic_optuna_front", "sum"),
        n_protocol=("protocol_optuna_front", "sum"),
    )
)
display(optuna_search_efficiency_df)

,track_id,state,status,n_trials
0,E1,COMPLETE,acceptable,100
1,E2,COMPLETE,acceptable,100
2,E4,COMPLETE,acceptable,84
3,E4,COMPLETE,calculable_but_not_acceptable,16
4,E5,COMPLETE,acceptable,100
5,E6,COMPLETE,acceptable,100
6,E7,COMPLETE,acceptable,79
7,E7,COMPLETE,calculable_but_not_acceptable,21
8,E8,COMPLETE,acceptable,83
9,E8,COMPLETE,calculable_but_not_acceptable,17


,track_id,study_scope,n_diagnostic,n_protocol
0,E1,protocol,6,6
1,E2,protocol,68,68
2,E4,diagnostic_only,76,0
3,E5,protocol,1,1
4,E6,protocol,10,10
5,E7,protocol,4,4
6,E8,diagnostic_only,83,0


,evaluation_track,track_id,decision_mode,study_name,study_scope,eligibility_status,study_status,n_domain_configurations,trial_budget,n_trials,n_complete_trials,n_pruned_trials,...,duplicate_trial_rate,domain_coverage_rate,pareto_reference_scope,n_exhaustive_pareto_configurations,n_exhaustive_pareto_recovered,exhaustive_pareto_recall,uniform_recall_expectation,pareto_recall_delta_vs_uniform,pareto_recall_lift_vs_uniform,budget_status,optuna_conclusion,exhaustive_reference_retained
0,object_train__object_projection__2way,E1,2way,04B_E1_non_noisy_all_d55c183926ce,protocol,eligible,runnable,10,100,100,100,0,...,0.90,1.000000,protocol_pareto_front,6,6,1.000000,0.999973,2.656140e-05,1.000027,sufficient,neutral,True
1,object_train__object_projection__3way,E2,3way,04B_E2_non_noisy_all_d55c183926ce,protocol,eligible_with_warning,runnable,402,100,100,100,0,...,0.15,0.211443,protocol_pareto_front,327,67,0.204893,0.220472,-1.557861e-02,0.929340,insufficient,insufficient,True
2,object_train__pixel_projection__2way,E3,2way,04B_E3_non_noisy_all_d55c183926ce,unsupported_empty,unsupported_internal_calibration,not_runnable_no_domain,0,100,0,0,0,...,NaN,NaN,protocol_pareto_front,0,0,NaN,NaN,NaN,NaN,not_estimable,not_estimable,True
3,object_train__pixel_projection__3way,E4,3way,04B_E4_non_noisy_all_d55c183926ce,diagnostic_only,unsupported_domain_shift,runnable_diagnostic_only,220,100,100,100,0,...,0.21,0.359091,diagnostic_pareto_front,205,75,0.365854,0.365921,-6.729515e-05,0.999816,insufficient,insufficient,True
4,pixel_train__object_projection__2way,E5,2way,04B_E5_non_noisy_all_d55c183926ce,protocol,eligible,runnable,1,100,100,100,0,...,0.99,1.000000,protocol_pareto_front,1,1,1.000000,1.000000,0.000000e+00,1.000000,sufficient,neutral,True
5,pixel_train__object_projection__3way,E6,3way,04B_E6_non_noisy_all_d55c183926ce,protocol,eligible,runnable,10,100,100,100,0,...,0.90,1.000000,protocol_pareto_front,10,10,1.000000,0.999973,2.656140e-05,1.000027,sufficient,neutral,True
6,pixel_train__pixel_projection__2way,E7,2way,04B_E7_non_noisy_all_d55c183926ce,protocol,eligible_with_warning,runnable,7,100,100,100,0,...,0.93,1.000000,protocol_pareto_front,4,4,1.000000,1.000000,2.019859e-07,1.000000,sufficient,neutral,True
7,pixel_train__pixel_projection__3way,E8,3way,04B_E8_non_noisy_all_d55c183926ce,diagnostic_only,unsupported_domain_shift,runnable_diagnostic_only,397,100,100,100,0,...,0.16,0.211587,diagnostic_pareto_front,381,80,0.209974,0.222916,-1.294234e-02,0.941941,insufficient,insufficient,True


## E — Gel prospectif du plan d'ablation

Les références viennent du front protocolaire exhaustif 04A, jamais du seul sous-ensemble visité par Optuna. Les paires exactes ne diffèrent que par les facteurs déclarés. Une chaîne spectrale absente du domaine reste explicitement `unsupported_no_valid_counterpart`. Les sensibilités de seuil sont perturbées autour des valeurs verrouillées sans réoptimisation.

In [6]:
preregistered_ablation_plan_df = build_preregistered_ablation_plan(
    grid_configurations_df,
    grid_pareto_reference_df,
    projection_eligibility_df,
    protocol_hash=upstream_protocol_hash,
    search_plan_hash=search_plan_hash,
    spatial_lock=spatial_lock,
)
if preregistered_ablation_plan_df.empty:
    raise RuntimeError("The preregistered ablation plan is empty.")
if not preregistered_ablation_plan_df["preregistered"].all():
    raise RuntimeError("Every frozen ablation row must be preregistered for the next v3 run.")
if not preregistered_ablation_plan_df["search_plan_hash"].eq(search_plan_hash).all():
    raise RuntimeError("Ablation plan search hash mismatch.")
display(
    preregistered_ablation_plan_df.groupby(
        ["contrast_type", "plan_status"], as_index=False
    ).agg(n_contrasts=("ablation_id", "count"))
)

,contrast_type,plan_status,n_contrasts
0,interaction,planned_four_cell_match_required,719
1,paired_variant,planned,2
2,paired_variant,unsupported_no_valid_counterpart,268
3,strict_ablation,planned,16
4,strict_ablation,unsupported_no_valid_counterpart,567
5,threshold_sensitivity,planned,1370


## F — Sauvegarde compacte et manifeste

In [7]:
score_column_audit_df = assert_no_forbidden_score_columns(
    {
        "optuna_trials": optuna_trials_df,
        "optuna_pareto_candidates": optuna_pareto_candidates_df,
        "optuna_search_efficiency": optuna_search_efficiency_df,
        "preregistered_ablation_plan": preregistered_ablation_plan_df,
    }
)
display(score_column_audit_df)

save_parquet(optuna_trials_df, OUTPUT_PATHS["trials"])
save_parquet(optuna_pareto_candidates_df, OUTPUT_PATHS["pareto_candidates"])
save_parquet(optuna_search_efficiency_df, OUTPUT_PATHS["search_efficiency"])
save_parquet(optuna_errors_df, OUTPUT_PATHS["errors"])
save_parquet(preregistered_ablation_plan_df, OUTPUT_PATHS["ablation_plan"])

output_hashes = {
    key: sha256_file(OUTPUT_PATHS[key])
    for key in ("trials", "pareto_candidates", "search_efficiency", "errors", "ablation_plan")
}
if STORAGE_PATH.exists():
    output_hashes["study_storage"] = sha256_file(STORAGE_PATH)

protocol = {
    "notebook": "04B_simca_optuna_search",
    "protocol_task_ids": [29, 30],
    "purpose": expcfg.SIMCA_OPTUNA_PURPOSE,
    "results_tag": RESULTS_TAG,
    "parent_protocol_hash": upstream_protocol_hash,
    "search_plan_hash": search_plan_hash,
    "input_sha256": source_hashes,
    "output_sha256": output_hashes,
    "evaluation_source": "04A_exact_internal_metrics_lookup_only",
    "calibration_batches": list(expcfg.INTERNAL_CALIBRATION_BATCHES),
    "forbidden_batches": list(expcfg.INTERNAL_CALIBRATION_FORBIDDEN_BATCHES),
    "batch3_or_batch4_loaded": False,
    "sampler": {
        "name": expcfg.SIMCA_OPTUNA_SAMPLER_NAME,
        "multivariate": bool(expcfg.SIMCA_OPTUNA_SAMPLER_MULTIVARIATE),
        "n_startup_trials": int(expcfg.SIMCA_OPTUNA_N_STARTUP_TRIALS),
    },
    "objective_contract": expcfg.SIMCA_OPTUNA_OBJECTIVE_SPECS,
    "suggested_parameters": ["domain_config_id"],
    "technical_prune_statuses": list(expcfg.SIMCA_OPTUNA_TECHNICAL_PRUNE_STATUSES),
    "calculable_not_acceptable_policy": "complete_trial_retained_outside_protocol_front",
    "uniform_recall_formula": "1-(1-1/N)^B",
    "minimum_pareto_recall": float(expcfg.SIMCA_OPTUNA_MIN_PARETO_RECALL),
    "uniform_delta_tolerance": float(expcfg.SIMCA_OPTUNA_UNIFORM_RECALL_DELTA_TOLERANCE),
    "weighted_score_used": False,
    "exhaustive_reference_always_retained": True,
    "studies": json.loads(study_registry_df.to_json(orient="records")),
    "ablation_plan": {
        "path": str(OUTPUT_PATHS["ablation_plan"]),
        "sha256": output_hashes["ablation_plan"],
        "registration_status": expcfg.SIMCA_ABLATION_REGISTRATION_STATUS,
        "execution_notebook": "05_simca_validation_robustness",
        "no_retuning": True,
    },
    "legacy_batch3_exposure_disclosure": (
        "Legacy 04C/05 outputs predate this 8-track plan. The plan is prospective "
        "for the next v3 execution, not a claim of first-ever batch-3 blinding."
    ),
    "software": {"python": platform.python_version(), "optuna": optuna.__version__},
}
OUTPUT_PATHS["protocol"].write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

print("Saved:")
for path in [*OUTPUT_PATHS.values(), STORAGE_PATH]:
    print(" -", path)

,table,n_columns,forbidden_score_columns,score_free
0,optuna_trials,24,,True
1,optuna_pareto_candidates,15,,True
2,optuna_search_efficiency,26,,True
3,preregistered_ablation_plan,23,,True


Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\optuna_trials.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\optuna_pareto_candidates.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\optuna_search_efficiency.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\optuna_errors.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\preregistered_ablation_plan.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v3_non_noisy_all\optuna_protocol.json
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tra

## Contrat de sortie

- `optuna_trials.parquet` conserve chaque trial, ses doublons, son état, ses erreurs et sa durée.
- `optuna_pareto_candidates.parquet` conserve des identifiants compacts et les fronts diagnostique/protocolaire ; les configurations complètes restent dans 04A.
- `optuna_errors.parquet` contient uniquement les échecs techniques.
- `optuna_search_efficiency.parquet` possède exactement une ligne par track, y compris E3 sans domaine.
- `preregistered_ablation_plan.parquet` fige les paires, sensibilités et interactions autorisées avant la prochaine exécution v3 du batch 3.
- `optuna_protocol.json` relie les hashes 03C/04A, le plan de recherche, les huit études et le plan d'ablation.

04B n'élimine aucun candidat exhaustif non visité et ne choisit aucun modèle final.